# 3. Sample model

The diagram shows the objects of `shapes.project` and how they fit together. An arrow goes from an object to one it holds, labelled with the attribute, such as `project.sample_model`. The arrows from the library show what is loaded from it. The EasyShapes GUI app works on the same objects, on its Project, Sample Model and Analysis pages. Notebooks 1 to 3 use the parts on their own; notebook 4 puts them together in a project.

This notebook works with a vesicle in a buffer of water and salt, put together as a sample model.

```
                                   ┌─────────┐
                                   │ Project │
                                   └────┬────┘
               ┌────────────────────────┴────────────────────────┐
          sample_model                                        analysis
               ▼                                                 ▼
        ╔═════════════╗                                    ┌──────────┐
        ║ SampleModel ║                                    │ Analysis │
        ╚══════╤══════╝                                    └─────┬────┘
      ┌────────┴─────────┐                                       │
  structure            buffer                              equilibration
      │                  ▼                                       ▼
      │              ╔════════╗                          ┌───────────────┐
      │              ║ Buffer ║                          │ Equilibration │
      │              ╚═══╤════╝                          └───────┬───────┘
      │           ┌──────┴──────┐                       ┌────────┴─────────┐
      │        solvent      components            configuration          stages
      │           ▼             ▼                       ▼                  ▼
      │      ╔═════════╗     ╔══════╗       ┌───────────────────────┐  ┌───────┐
      │      ║ Solvent ║     ║ Salt ║       │ AnalysisConfiguration │  │ Stage │
      │      ╚════▲════╝     ╚══╤═══╝       └───────────▲───────────┘  └───────┘
      │        Library    cation, anion              Library
      │                         ▼
      │                      ╔═════╗
      │                      ║ Ion ║
      │                      ╚══▲══╝
    one of                   Library
      └────┬────────────────────────┬─────────────────────┐
           ▼                        ▼                     ▼
┌─────────────────────┐        ┌─────────┐        ╔════════════════╗
│ Ring / Rod /        │◄─unit──┤ Lattice ├──unit─►║ Ball / Vesicle ║
│ Bilayer / Monolayer │        └─────────┘        ╚═╤═══════════╤══╝
└──────────┬──────────┘                             │    layers / lamellae
           │                                        │           ▼
           │                                        │  ╔═════════════════╗
           │                                        │  ║ Layer / Lamella ║
           └───────────────────┬────────────────────┘  ╚═════════════════╝
                           components
                               ▼
                     ╔═══════════════════╗   load   ┌─────────┐
your file, SMILES ──►║ Component         ║◄─────────┤ Library │
                     ║ Surfactant, Lipid ╟─────────►│         │
                     ╚═════════╤═════════╝   save   └─────────┘
                          counter_ion
                               ▼
                            ┌─────┐
                            │ Ion │
                            └──▲──┘
                            Library
```

A sample model is a structure in its buffer: a solvent, and maybe a salt. Here we build an asymmetric DOPC/POPC vesicle in TIP3 water with 0.15 mol/L NaCl.

In this notebook you will:

1. load the components from the library
2. set up the vesicle and its leaflets
3. set up the buffer
4. assemble the sample model, which runs GROMACS
5. look at the model and export it

We have to start with a few setup cells. On `Colab` it installs GROMACS and Shapespyer, which takes about 3 minutes. If you run it locally, make sure to install Shapespyer in your virtual environment, and GROMACS on your system (on WSL or Linux: `sudo apt install gromacs`). [Shapespyer installation docs](https://simnavi.gitlab.io/shapespyer.io/installation.html)

In [ ]:
# @title Setup: on Colab, install GROMACS and Shapespyer (about 3 min) { display-mode: "form" }
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
CONFIG_FILE = Path("shapespyer.yaml")  # where Shapespyer keeps its files

if IN_COLAB:
    !apt-get -qq update > /dev/null && apt-get -qq install -y --no-install-recommends gromacs > /dev/null
    %pip install -q "shapespyer==0.3.0"

    # nglview draws molecules through Colab's widget manager
    from google.colab import output
    output.enable_custom_widget_manager()

    # the library, Temp and Projects go to /content, to browse them in the Files panel
    CONFIG_FILE.write_text("app_path: /content/shapespyer\n")
elif not CONFIG_FILE.is_file():
    err_msg = f"No {CONFIG_FILE} in {Path.cwd()}; run the notebook in its own folder, next to shapespyer.yaml"
    raise FileNotFoundError(err_msg)

The next cell provides helper functions for us to use with Colab. It provides functions that only show and download results.
These helper functions, as well as the helpers in other notebooks, **are not part of the Shapespyer workflow**. They are all in the `ColabHelpers` class, and the notebook always calls them from it, as `ColabHelpers.<name>()`, so you can tell them apart from the Shapespyer package.

In [ ]:
# @title ColabHelpers (not part of the Shapespyer workflow) { display-mode: "form" }
import shutil
from pathlib import Path

import mdtraj
import nglview


class ColabHelpers:
    """Workshop helpers, not part of Shapespyer: call them as ColabHelpers.<name>()."""

    @staticmethod
    def check_setup(config):
        """Stop with a message if GROMACS can't run; else print its version."""
        if config.gmx_problem is not None:
            err_msg = (
                f"This notebook needs GROMACS: {config.gmx_problem}. Install it, or set gmx_executable"
                f" in {config.file.name}. On Windows, the easy way is to run this notebook in WSL or on Colab."
            )
            raise RuntimeError(err_msg)
        print("GROMACS:", config.gmx_version)

    @staticmethod
    def show_structure(component):
        """Draw a component, as its .show() does; Shapespyer 0.3.0's .show() fails on Colab."""
        view = nglview.show_structure_file(str(component.structure_file_path))
        view.clear_representations()
        view.add_representation("ball+stick", selection=component.name, radius=0.25)
        return view

    @staticmethod
    def _file_with_suffix(asset, suffix):
        found = [path for path in asset.paths if path.suffix == suffix]
        if not found:
            raise FileNotFoundError(f"{asset.name} holds no {suffix} file yet")
        return found[0]

    @staticmethod
    def _molecules(topology, resname):
        """Atom indices of the residues named resname (a name or a list of names), or of all but the water."""
        # mdtraj renames TIP3 to HOH on loading, so select it with the "water" keyword
        if resname is None:
            return topology.select("not water")
        names = [resname] if isinstance(resname, str) else list(resname)
        atoms = topology.select(" or ".join(f"resname {name}" for name in names))
        if len(atoms) == 0:
            found = sorted({residue.name for residue in topology.residues})
            raise ValueError(f"No residues named {', '.join(names)}; the file has {', '.join(found)}")
        return atoms

    @staticmethod
    def show_without_water(asset, resname=None):
        """Draw the .gro an asset holds, such as a sample model or a stage, without the water.

        Give resname, a residue name or a list of them, to draw only those molecules.
        """
        frame = mdtraj.load(ColabHelpers._file_with_suffix(asset, ".gro"))
        shown = frame.atom_slice(ColabHelpers._molecules(frame.top, resname))
        print(f"{frame.n_atoms} atoms, box {frame.unitcell_lengths[0]} nm; {shown.n_atoms} shown")
        return nglview.show_mdtraj(shown)

    @staticmethod
    def list_files(folder):
        """Print the files under a folder, relative to it."""
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                print(path.relative_to(folder))

    @staticmethod
    def download_as_zip(folder):
        """Zip a folder into <folder>.zip next to it; on Colab, also download the zip."""
        archive = Path(shutil.make_archive(str(folder), "zip", folder))
        if IN_COLAB:
            from google.colab import files
            files.download(str(archive))
        return archive

In the setup we pass the `shapespyer.yaml` config file, which says where Shapespyer keeps its files. On `Colab` it is necessary, but in the common scenario, users won't have to use the config file, as the `shapes.project` package manages files automatically. Use the config if you want manual control over where the files are kept.

In [ ]:
# @title Config file, log level and work folder { display-mode: "form" }
from shapes.basics.utils import LogConfiguration
from shapes.project.files import AppFileManager

# the first call sets where the files go, for the whole session
config = AppFileManager(CONFIG_FILE).config
print("Workshop files:", config.app_path)
ColabHelpers.check_setup(config)

LogConfiguration("WARNING")

# the export and its zip go to a work folder, which the clean up deletes
WORK = config.app_path / "work"

In [ ]:
from shapes.basics.defaults import Fill
from shapes.project import Ion, Library, Lipid, SampleModel, Vesicle
from shapes.project.species import Buffer, Salt, Solvent

### The library

Here you choose where the library keeps its files: both the assets downloaded from the Shapespyer assets and the components you save yourself. We give `LocalAssets/` in the application directory, which is also the default. Only the first `Library()` of a session sets the path: later calls return the same library. `clean_temp()` deletes the working files that earlier runs left in Temp.

In [ ]:
library = Library(path=config.app_path / "LocalAssets")
library.clean_temp()

### Load the components

`Lipid.load(name)` takes a lipid from the library. A lipid holds the same fields as the surfactant of notebook 1: `name`, `paths`, `counter_ion`, `mint` and `mext`. Here we keep them as the library has them.

In [ ]:
dopc = Lipid.load("DOPC")
popc = Lipid.load("POPC")
ColabHelpers.show_structure(dopc)

In [ ]:
ColabHelpers.show_structure(popc)

### Set up the vesicle

A vesicle holds, as in notebook 2:

- `name`: the name of the structure, also the name of its `.gro` file.
- `components`: the components it is built from. A component added to the structure joins every layer.
- `fill`: how the molecules are placed on the sphere, a `Fill`: on rings (`Fill.RINGS0`, the default, or `Fill.RINGS`) or on a Fibonacci spiral (`Fill.FIBO`). We use `Fill.FIBO`.
- `fxz`: flattens each molecule in the XZ plane first, so it spreads less along Y.
- `rev`: swaps the interior and the exterior atoms of each molecule.
- `lamellae`: the concentric lamellae, one to start with; `lamellae.add()` adds one. The next cell sets up ours.

In [ ]:
vesicle = Vesicle(
    name="DemoVesicle",
    fill=Fill.FIBO,
    fxz=False,
    rev=False,
)
vesicle.components.append(dopc)
vesicle.components.append(popc)

print(vesicle)

### One lamella with different leaflets

A lamella holds:

- `rmin`: the radius of the empty space in the centre, where the inner leaflet starts, in nm. We use 5.
- `shell_thickness`: the distance from `rmin` to the outer end of the outer leaflet's bones, in nm: the thickness of the lamella. We use 5. With 0, the radius of the outer leaflet is scaled instead.
- `inner_leaflet_dmin` and `outer_leaflet_dmin`: the minimum distance between the bone atoms in each leaflet, in nm. We use 1.1 and 2.
- `symmetric`: whether the two leaflets share one composition. We set it to `False`, to give each leaflet its own.
- `inner_leaflet_composition` and `outer_leaflet_composition`: how much of each component each leaflet holds.
- `swap()`: exchanges the compositions of the two leaflets.

A composition holds one entry per component: whether it is present, and its molar ratio. A component joins present, with the ratio 1.

- `set_mol_ratio(key, value)`: sets the molar ratio of a component, given by itself or its name; 0 marks it absent.
- `set_presence(key, present)`: marks a component present or absent, keeping its molar ratio.
- `mol_percents`: the molar percentages of the components, in their order.

Here:

- inner leaflet: POPC and DOPC in the ratio 2 : 3
- outer leaflet: DOPC only (POPC is marked absent, so the 100 of DOPC makes 100 %)

Try:

- `lamella.swap()`, to put POPC outside
- `rmin` from 3 to 8 nm

In [ ]:
lamella = vesicle.lamellae[0]
lamella.symmetric = False
lamella.rmin = 5.0
lamella.shell_thickness = 5.0

lamella.inner_leaflet_composition.set_mol_ratio("POPC", 2.0)
lamella.inner_leaflet_composition.set_mol_ratio("DOPC", 3.0)
lamella.inner_leaflet_dmin = 1.1

lamella.outer_leaflet_composition.set_presence("POPC", False)
lamella.outer_leaflet_composition.set_mol_ratio("DOPC", 100.0)
lamella.outer_leaflet_dmin = 2.0

print(vesicle.lamellae)
print(lamella.inner_leaflet_composition.mol_percents, lamella.outer_leaflet_composition.mol_percents)

### Set up the buffer

TIP3 water with 0.15 mol/L NaCl. DOPC and POPC are neutral, so no counter ions are needed.

A buffer holds:

- `name`: the name of the buffer.
- `solvent`: the solvent the model is solvated in, loaded from the library like a component: `Solvent.load(name)`. The library has TIP3 water and ETOH, ethanol.
- `components`: what is dissolved in the solvent, such as salts. `append()` adds one.

A salt holds:

- `name`: the name of the salt.
- `cation` and `anion`: the positive and the negative ion, or the name of one to load from the library.
- `cation_count` and `anion_count`: the number of cations and anions in one formula unit, 1 and 1 for NaCl.
- `concentration`: the molar concentration in the buffer, in mol/L. 0.15 mol/L is about the salt of the body.

A sample model holds:

- `name`: the name of the model.
- `structure`: the structure, here the vesicle.
- `buffer`: the buffer it is solvated in.

Try `concentration` from 0 to 0.5 mol/L. Too much salt for the box stops the assembly with `Only N of M ... fit into the box`.

In [ ]:
buffer = Buffer(name="DemoBuffer", solvent=Solvent.load("TIP3"))
buffer.components.append(
    Salt(
        name="NaCl",
        cation=Ion.load("NAT"),
        cation_count=1,
        anion=Ion.load("CLA"),
        anion_count=1,
        concentration=0.15,
    )
)

model = SampleModel(name="DemoModel", structure=vesicle, buffer=buffer)
print(model)

### Assemble the sample model

`.assemble()` builds the vesicle, puts in the salt ions with `gmx insert-molecules` and fills the box with water with `gmx solvate`. Then the model holds its `.gro`, its `.top` and `assembly.log`.

If you change any setting of the model or the vesicle, call `.assemble()` again.

In [ ]:
model.assemble()
model.paths

### Look at the model

We hide the water, so the vesicle and the ions are easy to see.

In [ ]:
ColabHelpers.show_without_water(model)

### Export and download

`export_to()` copies the files of the model, and of every asset in it, into a folder you choose, here `DemoModel/` in the work folder. That keeps the model to reuse: `SampleModel.import_from()` loads it back from the folder. The vesicle is assembled on its own too, so the export also holds the vesicle's own `.gro`. The cell after it zips the export; on Colab it also downloads the zip.

In [ ]:
vesicle.assemble()
model.export_to(WORK / "DemoModel", overwrite=True)
ColabHelpers.list_files(WORK / "DemoModel")

In [ ]:
ColabHelpers.download_as_zip(WORK / "DemoModel")

### Clean up (optional)

This deletes the work folder, with the export and its zip, and Temp, where the vesicle and the model were assembled. `force=True` is needed for Temp, because the assets of this session still hold their files there.

In [ ]:
import shutil

shutil.rmtree(WORK, ignore_errors=True)
library.clean_temp(force=True)
print(WORK.exists())